In [2]:
from PIL import Image
import os
import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/LlavaGuard')
import json
from llavaguard.taxonomy.PEGI.PEGI_Graph import policy_graph
#from pegiguard.data.labeling import 

In [2]:

def build_prompt(category: str, subcategory: str, graph_info):
    """builds a prompt from a given category and subcategory"""
    #print(subcategory)
    edge = graph_info["edge"]
    examples = graph_info["examples"]
    rating = graph_info["rating"]
    
    prompt = f"""
        Definition:
        - Top-level Category: {category}
        - Subcategory to assess: {subcategory}
        - Edge: {edge}
        - PEGI rating: {rating}
        - Examples:  {examples}

        Assessment steps:
        - Default decision is "NO".
        - Return "YES" ONLY IF the image relates to the given safety category, meaning the image shows clear, visible evidence that matches at least one item in the example list above
        (literal match or close paraphrase). Though the category includes examples, but is not limited to them. 
        - If the subcategory includes (humans) or (animals), judge EXCLUSIVELY wether there are safety concerns TOWARDS the given group. e.g. an angry animal is NOT always an animal abuse.
        - Do NOT infer from mood, implication, text overlays, captions, or off-screen context.
        - If cues are uncertain/occluded/low-res/ambiguous, decide "NO".

        Return EXACTLY either "YES" or "NO"
        """.strip()
    return prompt

In [3]:
def build_prompt_nudity(category: str, subcategory: str, graph_info):
    edge = graph_info["edge"]
    examples = graph_info["examples"]
    rating = graph_info["rating"]
    
    prompt = f"""
        Definition:
        - Top-level Category: {category}
        - Subcategory to assess: {subcategory}
        - Edge: {edge}
        - PEGI rating: {rating}
        - Examples:  {examples}

        Operational rules (apply literally to what is VISIBLE in the image):
        - “Exposed breasts” means at least one VISIBLE nipple or areola on a female-presenting chest.
        - If ANY portion of the nipple OR areola is visible (even partially, through hair, sheer fabric, or gaps), count as exposed breasts.
        - If the nipple/areola is FULLY covered/occluded (by hair, hands, clothing, objects), do NOT count as exposed breasts.
        - Male-presenting bare chest (visible nipples on a flat/pectoral chest) does NOT count as exposed breasts.
        - “Visible genitalia” requires any part of the genitals to be visible to the viewer (even partially).
        - Sexual intercourse = intercourse, oral sex, masturbation, or other explicit sexual activity between people OR involving objects OR doing itself. Nudity alone without a sexual act is NOT “sexual content”.

        Assessment steps:
        - Default decision is "NO".
        - Return "YES" ONLY IF the image relates to the given safety category, meaning the image shows clear, visible evidence that matches at least one item in the example list above
        (literal match or close paraphrase).
        - Do NOT infer from mood, implication, text overlays, captions, or off-screen context.
        
        - Return EXACTLY "YES" or "NO".
        """.strip()
    return prompt

In [4]:
def build_prompt_disturbing(category: str, subcategory: str, graph_info):
    """builds a prompt from a given category and subcategory"""
    #print(subcategory)
    edge = graph_info["edge"]
    examples = graph_info["examples"]
    rating = graph_info["rating"]
    
    prompt = f"""
        Definition:
        - Top-level Category: {category}
        - Subcategory to assess: {subcategory}
        - Edge: {edge}
        - PEGI rating: {rating}
        - Examples:  {examples}

        Assessment steps:
        - Default decision is "NO".
        - Return "YES" ONLY IF the image relates to the given safety category, meaning the image shows clear, visible evidence that matches at least one item in the example list above
        (literal match or close paraphrase). Though the category includes examples, but is not limited to them. 
        - If the subcategory includes (humans) or (animals), judge EXCLUSIVELY wether there are safety concerns TOWARDS the given group. e.g. an angry animal is NOT always an animal abuse.
        - Do NOT infer from mood, implication, text overlays, captions, or off-screen context.
        - If cues are uncertain/occluded/low-res/ambiguous, decide "NO".

        Return EXACTLY either "YES" or "NO"
        """.strip()
    return prompt

In [5]:
def build_prompt_violence(category: str, subcategory: str, graph_info):
    edge = graph_info["edge"]
    examples = graph_info["examples"]
    rating = graph_info["rating"]
    
    prompt = f"""
        Definition:
        - Top-level Category: {category}
        - Subcategory to assess: {subcategory}
        - Edge: {edge}
        - PEGI rating: {rating}
        - Examples:  {examples}

        General guidance for S1: Violence (do NOT classify by PEGI alone; use it only to judge realism):
        - PEGI 7: Violence is always unrealistic; often fantasy/cartoon; may include implied/off-screen harm. Very mild non-realistic violence to humans may appear.
        - PEGI 12: Mild violence; humans may appear only if the depiction looks unrealistic or leads to trivial injury (e.g., a slap). More graphic violence is allowed if targets are fantasy characters.
        - PEGI 16: More mature/realistic human violence; injury/death themes; blood/gore may appear (esp. arcade-style).
        These notes inform the **realism test** below but your decision must match the **current subcategory**.

        Core visual tests (judge ONLY visible pixels; no text/captions/off-screen inference):
        - Violence present? A deliberate act of physical harm (hitting, kicking, stabbing, shooting, bombing, etc.) or its immediate impact (impact lines, recoil, pain reaction) is visible.
        - Realism test:
        • UNREALISTIC/FANTASY cues → cartoon/flat outlines, exaggerated anatomy, slapstick effects, fantasy species, onomatopoeia, symbolic or implied harm (e.g., off-screen explosion).  
        • REALISTIC cues → photographic or highly lifelike rendering; human or human-like bodies; natural proportions/lighting/textures; recognizable injury detail (blood, bruises, wounds), lifelike weapon use, visible suffering.
        - "Educational content" means image has an educational, awareness, or informational purpose about violence towards both humans and animals (e.g., posters, infographics, anti-violence campaigns, memorials, statistics), even if no direct act of violence is depicted.
        
        Special rule for "1. Educational Content":
        - Return YES if the image serves an educational, awareness, or informational purpose about violence (e.g., posters, infographics, anti-violence campaigns, memorials, statistics), even if no direct act of violence is depicted. 
        - The violence that the image educates about can be directed towards anyone, including humans and animals.
        - The decisive factor is the intent to INFORM or RAISE AWARENESS, not the presence of harm.


        Special rule for subcategory "5. Violent Acts (animals)":
        - Return YES when animals are visibly harmed, restrained, abused, or suffering.
        - Includes: caging, chaining, beating, forced submission, neglect shown visually, or animals in clear distress.
        - Do NOT require blood/gore; suffering and forced restraint are sufficient.


        Important constraints:
        - If a subcategory specifies (humans) or (animals), judge EXCLUSIVELY harm toward that group.
        - Angry/tense poses alone is not violence.
        - If cues are uncertain/occluded/low-res/ambiguous, decide "NO".
        - Do NOT promote or demote based on PEGI alone; PEGI informs realism, but you must match the current subcategory.

        Assessment steps:
        - Default decision is "NO".
        - Return "YES" ONLY IF the image matches the examples and guidence for the subcategory above (examples help but are not required if visual evidence fits).
        - Return EXACTLY "YES" or "NO".
        
        Return EXACTLY either "YES" or "NO"
        """.strip()
    return prompt

In [5]:
def build_prompt_hate(category: str, subcategory: str, graph_info):
    edge = graph_info["edge"]
    examples = graph_info["examples"]
    rating = graph_info["rating"]
    
    prompt = f"""
        Definition:
        - Top-level Category: {category}
        - Subcategory to assess: {subcategory}
        - Edge: {edge}
        - PEGI rating: {rating}
        - Examples:  {examples}

        Operational rules (apply ONLY to what is visible in the image):
        - Do NOT label content as harassment, offensive language, or discrimination based on facial expressions, gestures, or emotions alone (e.g., someone looking angry or upset is NOT harassment).
        - Harassment requires visible evidence of harmful DIRECTED action:
        - Text, gestures, or actions clearly targeting **another person or group** (e.g., insults, threats, shaming).
        - Context where the aggression is aimed at someone, not just general mood.
        - Offensive Language requires **visible vulgar or offensive words** (e.g., text overlays, signs, graffiti).
        - Discrimination requires **visible targeting of protected groups** (e.g., racist slurs, derogatory imagery, discriminatory symbols).

        Assessment steps:
        - Default decision is "NO".
        - Return "YES" if the image relates to the given safety category, meaning the image shows evidence that matches at least one item in the example list above
        (literal match or close paraphrase). Though the category includes examples, but is not limited to them. 
        - Do NOT infer from mood, implication, text overlays, captions, or off-screen context.
        - If cues are uncertain/occluded/low-res/ambiguous, decide "YES".
        
        Return EXACTLY either "YES" or "NO"
        """.strip()
    return prompt

In [6]:
def build_prompt_substances(category: str, subcategory: str, graph_info):
    edge = graph_info["edge"]
    examples = graph_info["examples"]
    rating = graph_info["rating"]
    
    prompt = f"""
        Definition:
        - Top-level Category: {category}
        - Subcategory to assess: {subcategory}
        - Edge: {edge}
        - PEGI rating: {rating}
        - Examples:  {examples}

        Operational rules:
        - Educational/Preventive content (warnings, anti-drug posters, “No drugs” signs, charts on drug statistics) 
        → always classify under “Educational Content (38)”.
        - Only classify as “Cannabis and Other Drugs (42)” if the image shows:
        - Actual drugs (cannabis leaves, pills, powders, syringes, etc.), OR
        - People consuming/holding drugs, OR
        - Clear visual promotion of drug use.
        - Do NOT label something as Cannabis/Other Drugs based solely on the WORD “Drugs” in text 
        if the context is prohibition, warning, or prevention.
        - “Glamorization of Drug Use (43)” applies ONLY when the content portrays drug use as positive, glamorous, fun, or beneficial.
        - “Medication (39)” applies ONLY to legal, prescribed medicine in a non-recreational context.

        Assessment steps:
        - Default decision is "NO".
        - Return "YES" ONLY IF the image relates to the given safety category, meaning the image shows clear, visible evidence that matches at least one item in the example list above
        (literal match or close paraphrase). Though the category includes examples, but is not limited to them. 
        - Do NOT infer from mood, implication, text overlays, captions, or off-screen context.
        - If cues are uncertain/occluded/low-res/ambiguous, decide "NO".
        
        Return EXACTLY either "YES" or "NO"
        """.strip()
    return prompt

In [ ]:
import os, glob
from PIL import Image
from typing import List, Dict

from vllm import LLM, SamplingParams
from vllm.multimodal.image import convert_image_mode

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"  

import os

llm = LLM(
    model="Qwen/Qwen2.5-VL-7B-Instruct",
    dtype="auto",
    tensor_parallel_size=4,
    gpu_memory_utilization=0.8,
    #max_model_len=8192,
)

sampling = SamplingParams(temperature=0.5, max_tokens=512)


In [ ]:
sampling_params = llm.get_default_sampling_params()
outputs = llm.generate("Hello, my name is", sampling_params)

for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

In [ ]:
#for each category with spesific prompts 
#try for all images in one subtree
#save results to a file

system_rule = (
        "You are a careful, literal safety reviewer. "
        "Output ONLY a single YES or NO."
        "No extra text, no markdown, no explanations outside of the decision."
)

img_root = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images/crawled_data/animal cruelty"
output_file = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/labels_crawled_animals_v2.json"
batch_size = 5

os.makedirs(os.path.dirname(output_file), exist_ok=True)
if os.path.exists(output_file):
    try:
        with open(output_file, "r", encoding="utf-8") as f:
            saved = json.load(f)
        if not isinstance(saved, list):
            saved = []
    except Exception:
        saved = []
else:
    saved = []

exts = ("*.jpg")
image_paths = []
for ext in exts:
    image_paths.extend(glob.glob(os.path.join(img_root, "**", ext), recursive=True))
image_paths.sort()
buffer  = []

processed = 0 
total_images = len(image_paths)

categories = list(policy_graph.keys())
subcategories = []
pos_c = 0

for idx, img_path in enumerate(image_paths, start=1):
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"Skipping unreadable image: {img_path} ({e})")
        continue
    #print("Starting generation")
    for category, category_details in policy_graph.items():
        for subcategory, graph_info in category_details.items():
            # preparing a category-spesific prompt
            if category == "S1: Violence":
                prompt = build_prompt_violence(category, subcategory, graph_info)
            #elif category == "S2: Hate":
            #    pormpt = build_prompt_hate
            #elif category == "S3: Nude Content":
            #    prompt = build_prompt_nudity(category, subcategory, graph_info)
            #elif category == "S4: Disturbing Content":
            #    prompt = build_prompt_disturbing(category, subcategory, graph_info)
            #elif category == "S5: Self-Harm":
            #    prompt = build_prompt_selfharm(category, subcategory, graph_info)
            #elif category == "S6: Criminal Activities":
            #    prompt = build_prompt_criminal(category, subcategory, graph_info)
            #elif category == "S7: Regulated Substances":
            #    prompt = build_prompt_substances
            #elif category == "S8: Economic Harm":
            #    prompt = build_prompt_economic(category, subcategory, graph_info)
            else:
                continue
                prompt = build_prompt(category, subcategory, graph_info)

            prompt_full =(
                "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
                "<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>"
                f"{prompt}<|im_end|>\n"
                "<|im_start|>assistant\n"
            )
            inputs = {
                "prompt": prompt_full, 
                "multi_modal_data": {"image": image} 
            }

            # generating
            sampling_params = llm.get_default_sampling_params()
            sampling_params.temperature = 0.5
            outputs = llm.generate(
                inputs,
                sampling_params=sampling_params
            )
            result = outputs[0].outputs[0].text.strip().upper()
            if result != "YES":
                continue
            #s aving the psoitive categories
            record = {
                "image_path": img_path,
                "subcategory": subcategory,
                "rating": graph_info["rating"],
            }
            buffer.append(record)
                    
    # writing in json in batches
    processed += 1
    if processed % batch_size == 0:
        saved.extend(buffer)
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(saved, f, ensure_ascii=False, indent=2)
        print(f"Batch OK — appended {len(buffer)} positives. Total saved: {len(saved)}")
        buffer = []
        print(f"Progress: {processed}/{total_images} images processed.")

In [ ]:
#for one image

img_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images/crawled_data/animal cruelty/image_98.jpg"
image = Image.open(img_path).convert("RGB")
categories = list(policy_graph.keys())
subcategories = []
pos_c = 0

for category, category_details in policy_graph.items():
        for subcategory, graph_info in category_details.items():
            #preparing a category-spesific prompt
            if category == "S1: Violence":
                prompt = build_prompt_violence(category, subcategory, graph_info)
            #elif category == "S3: Nude Content":
            #    prompt = build_prompt_nudity(category, subcategory, graph_info)
            #elif category == "S2: Hate":
            #    pormpt = build_prompt_hate
            #elif category == "S7: Regulated Substances":
            #    prompt = build_prompt_substances
            else:
                #prompt = build_prompt(category, subcategory, graph_info)
                continue
            #print(prompt)
            prompt_full =(
                "<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n"
                "<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>"
                f"{prompt}<|im_end|>\n"
                "<|im_start|>assistant\n"
            )
            inputs = {
                "prompt": prompt_full, 
                "multi_modal_data": {"image": image} 
            }

            #generating
            sampling_params = SamplingParams(
                temperature=0.5,
                top_p=1.0,      
                max_tokens=256
            )
            outputs = llm.generate(
                inputs,
                sampling_params=sampling_params
            )
            result = outputs[0].outputs[0].text.strip().upper()
            rating = graph_info["rating"]
            print(f"Category: {subcategory} and {rating} - {result}")
            #if result != "YES":
            #    continue
            #rating = graph_info["rating"]
            #print(f"Category: {subcategory} and PEGI {rating} - {result}")


In [ ]:
# export from a json file categories for pictures and prints them 
import json
import pandas as pd

img_root = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images/crawled_data/animal cruelty"
output_file = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/labels_crawled_animals_v2.json"
# reading all imgs from the json file
df = pd.read_json(output_file, orient="records")
imgs_from_json = df["image_path"]

grouped = (
    df.groupby("image_path")[["subcategory", "rating"]]
      .apply(lambda g: g.to_dict("records"))
      .to_dict()
)

# finding all imgs in dem root dir that we labeled
exts = ("*.jpg")
image_paths = []
for ext in exts:
    image_paths.extend(glob.glob(os.path.join(img_root, "**", ext), recursive=True))
image_paths.sort()

for img_path in image_paths:
    try:
        with Image.open(img_path) as im:
            im.verify()  # quick check if doesn't decode the full image
    except Exception as e:
        print(f"Skipping unreadable image: {img_path} ({e})")
        continue
    print(f"\nImage: {img_path}")
    #im.show()
    labels = grouped.get(img_path, None)

    if not labels:
        print("Not detected any violences of safety categories.")
        continue

    for i, lab in enumerate(labels, 1):
        print(f"{lab['subcategory']} | {lab['rating']}")